Up until now we only computed the distance.

But a real system needs to check if they are the same persons.

##### collapse

In [1]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(image_size=160, margin=0, device=device)

resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

In [3]:
img1 = Image.open('faces/single/shail/shail1.jpg')
img2 = Image.open('faces/single/amitabh_bachchan/ab1.png')

In [4]:
face1 = mtcnn(img1)
face2 = mtcnn(img2)

In [5]:
face1 = face1.unsqueeze(0).to(device)
face2 = face2.unsqueeze(0).to(device)

with torch.no_grad():
    emb1 = resnet(face1)
    emb2 = resnet(face2)

### Step 1 — Turn It Into A Function

In [6]:
def compare_faces(emb1, emb2, threshold=0.8):
    distance = torch.norm(emb1 - emb2).item()
    
    if distance < threshold:
        return True, distance
    else:
        return False, distance

In [7]:
result, dist = compare_faces(emb1, emb2)

print("Same Person?", result)
print("Distance:", dist)

Same Person? False
Distance: 1.2690349817276


### Why Threshold Is Important

here are two types of mistakes:

- False Positive → different people marked as same
- False Negative → same person marked as different

Security systems care more about:

False positives

Because that means someone else gets access.

So usually threshold is chosen slightly strict.

### Types of distance calculation algorithms

Right now we are using:

    Euclidean distance

But there is another common method:

    Cosine similarity

We will compare both.

### Cosine Similarity

In [8]:
import torch.nn.functional as F

def cosine_similarity(emb1, emb2):
    return F.cosine_similarity(emb1, emb2).item()

In [9]:
cos_sim = cosine_similarity(emb1, emb2)
print("Cosine Similarity: ", cos_sim)

Cosine Similarity:  0.19477519392967224


### What Cosine Similarity Means

Cosine similarity measures:

> Are these vectors pointing in the same direction?

It ignores magnitude.

Value range:

- 1 → identical direction
- 0 → unrelated
- -1 → opposite

For faces:

- Same person → usually ~0.6 to 0.9
- Different people → lower

Euclidean distance measures:

> Absolute separation in space

Cosine similarity measures:

> Angle between identity vectors

Many modern face systems normalize embeddings and use cosine.

Later we’ll see why.